In [44]:
# Import the libraries needed for data preparation and model building.
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

# I am using a pipeline so preprocessing and model training stay together.


In [45]:
df = pd.read_csv('customer_churn_1M.csv')

# Customer Churn Prediction

## 1. Understanding the problem
Customer churn means a customer stops using a company's service. In this project, I will use customer information to predict whether a customer is likely to churn.

The target column is `churn`. I will first inspect the dataset, handle missing values, prepare numerical and categorical columns, and then train a Logistic Regression model.

**Note:** I will evaluate more than accuracy because the churn and non-churn classes may not be balanced.


In [46]:
# Check the number of rows and columns.
# This helps me understand the size of the dataset.
df.shape

(1000000, 32)

## 2. Initial data inspection

I am checking the dataset structure and a few rows before making changes. This helps me identify the column names, data types, and whether the data looks as expected.


In [47]:
df.shape

(1000000, 32)

In [48]:
# Review column data types and non-null counts.
# This helps identify columns with missing values and their data types.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   customer_id                  1000000 non-null  object 
 1   signup_date                  1000000 non-null  object 
 2   age                          1000000 non-null  int64  
 3   gender                       1000000 non-null  object 
 4   annual_income                970041 non-null   float64
 5   education                    1000000 non-null  object 
 6   marital_status               1000000 non-null  object 
 7   dependents                   1000000 non-null  int64  
 8   tenure                       1000000 non-null  int64  
 9   contract                     1000000 non-null  object 
 10  payment_method               1000000 non-null  object 
 11  paperless_billing            1000000 non-null  object 
 12  senior_citizen               1000000 non-nu

In [49]:
# View the first five rows to get a quick look at the data.
df.head()

,customer_id,signup_date,age,gender,annual_income,education,marital_status,dependents,tenure,contract,...,has_streaming_tv,has_streaming_movies,customer_satisfaction,num_complaints,num_service_calls,late_payments,avg_monthly_gb,days_since_last_interaction,credit_score,churn
0,CUST0000000001,2022-12-12 12:53:58.199463,43,Female,55085.25,college,married,1,2,two_year,...,1,1,9.0,0.0,0,0,109.63,16,NaN,0
1,CUST0000000002,2022-01-13 12:53:58.199973,18,Male,60786.11,master,married,1,22,one_year,...,0,1,7.0,0.0,3,1,63.25,134,585.0,0
2,CUST0000000003,2023-09-04 12:53:58.199985,38,Female,73184.32,high_school,widowed,0,3,two_year,...,1,0,6.0,1.0,1,0,47.77,11,632.0,0
3,CUST0000000004,2022-06-27 12:53:58.199992,44,Male,40923.78,high_school,married,1,6,two_year,...,0,1,5.0,2.0,2,1,50.82,6,569.0,0
4,CUST0000000005,2022-12-08 12:53:58.199999,45,Female,36400.94,bachelor,single,0,9,two_year,...,0,1,8.0,1.0,1,0,16.74,18,657.0,0


In [50]:
# View the last five rows as another quick check of the loaded file.
df.tail()

,customer_id,signup_date,age,gender,annual_income,education,marital_status,dependents,tenure,contract,...,has_streaming_tv,has_streaming_movies,customer_satisfaction,num_complaints,num_service_calls,late_payments,avg_monthly_gb,days_since_last_interaction,credit_score,churn
999995,CUST0000999996,2025-02-03 12:54:09.482412,25,Male,21309.78,college,single,4,5,one_year,...,1,1,2.0,1.0,2,0,7.12,14,688.0,0
999996,CUST0000999997,2024-09-08 12:54:09.482417,22,Female,164236.89,college,married,1,11,one_year,...,0,0,5.0,0.0,3,0,0.00,1,653.0,0
999997,CUST0000999998,2021-09-13 12:54:09.482423,44,Female,34258.27,college,divorced,1,12,one_year,...,0,1,6.0,0.0,0,0,22.43,139,669.0,0
999998,CUST0000999999,2025-08-30 12:54:09.482428,65,Male,38687.93,high_school,single,0,24,one_year,...,1,1,5.0,0.0,0,0,0.00,17,843.0,0
999999,CUST0001000000,2023-02-14 12:54:09.482434,31,Male,23617.51,bachelor,single,1,3,one_year,...,1,1,7.0,0.0,1,1,NaN,85,683.0,0


In [51]:
# View five random rows instead of only looking at the beginning or end.
df.sample(5)

,customer_id,signup_date,age,gender,annual_income,education,marital_status,dependents,tenure,contract,...,has_streaming_tv,has_streaming_movies,customer_satisfaction,num_complaints,num_service_calls,late_payments,avg_monthly_gb,days_since_last_interaction,credit_score,churn
37144,CUST0000037145,2024-10-20 12:53:58.416249,30,Male,41860.69,college,single,1,6,one_year,...,0,1,9.0,0.0,1,0,21.98,71,NaN,0
147577,CUST0000147578,2025-09-03 12:53:59.606434,38,Male,44433.27,bachelor,single,0,60,one_year,...,1,0,7.0,1.0,0,1,11.23,1,627.0,0
359578,CUST0000359579,2025-04-04 12:54:02.041484,44,Female,47522.69,master,single,1,37,two_year,...,1,1,8.0,0.0,1,0,38.67,44,660.0,0
568085,CUST0000568086,2024-02-17 12:54:04.438811,61,Female,94257.59,high_school,married,0,9,two_year,...,1,0,7.0,2.0,2,2,47.00,3,779.0,1
765823,CUST0000765824,2024-12-15 12:54:06.815674,40,Male,21181.88,college,married,2,1,two_year,...,0,1,5.0,0.0,2,2,0.00,103,616.0,0


In [52]:
# Count all missing cells in the full dataset.
df.isnull().sum().sum()

np.int64(170193)

In [53]:
# Count missing values separately for each column.
# Columns with missing values will be handled in the preprocessing pipeline.
df.isnull().sum()

,0
customer_id,0
signup_date,0
age,0
gender,0
annual_income,29959
education,0
marital_status,0
dependents,0
tenure,0
contract,0


In [54]:
# Check whether any complete rows are duplicated.
df.duplicated().sum()

np.int64(0)

We dont have duplicated values , but have missing value in large amount and handlling this is very important

In [55]:
# Remove customer_id because it is an identifier, not a useful customer behaviour feature.
# I am also dropping signup_date for this first model rather than using the raw date text.
df.drop(columns=['customer_id', 'signup_date'], inplace=True)

In [56]:
# X contains the input features used for prediction.
# y contains the answer the model should learn to predict.
X = df.drop('churn', axis=1)
y = df['churn']

In [57]:
# Keep 20% of the data for testing and use the remaining 80% for training.
# Stratification keeps the churn/non-churn proportion similar in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [58]:
# Separate columns based on their data types.
# Numerical columns will be filled and scaled; text columns will be filled and encoded.
num_features = X.select_dtypes(include=['number']).columns.tolist()
cat_features = X.select_dtypes(exclude=['number']).columns.tolist()

In [59]:
num_features

['age',
 'annual_income',
 'dependents',
 'tenure',
 'senior_citizen',
 'monthlycharges',
 'totalcharges',
 'num_services',
 'has_phone_service',
 'has_internet_service',
 'has_online_security',
 'has_online_backup',
 'has_device_protection',
 'has_tech_support',
 'has_streaming_tv',
 'has_streaming_movies',
 'customer_satisfaction',
 'num_complaints',
 'num_service_calls',
 'late_payments',
 'avg_monthly_gb',
 'days_since_last_interaction',
 'credit_score']

In [60]:
cat_features

['gender',
 'education',
 'marital_status',
 'contract',
 'payment_method',
 'paperless_billing']

In [61]:
# Numerical preprocessing:
# - Median imputation is less affected by extreme values than the mean.
# - StandardScaler puts numerical features on a comparable scale.
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

# Categorical preprocessing:
# - Fill missing categories with the most common value in the training data.
# - OneHotEncoder converts text categories into numeric columns for the model.
# handle_unknown='ignore' avoids an error if a new category appears in test data.
cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoding', OneHotEncoder(handle_unknown='ignore'))
])

In [62]:
# Apply the numerical and categorical pipelines to their matching columns.
# The preprocessing steps are fitted on training data when the full pipeline is trained.
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
], remainder='drop')

In [63]:
# Logistic Regression is a useful baseline for a binary target such as churn (0 or 1).
# A higher max_iter gives the model more iterations to converge.
model = LogisticRegression(max_iter=1000)

In [64]:
# Combine preprocessing and the classifier into one pipeline.
# fit() learns preprocessing values and model parameters using only the training data.
pipe = Pipeline(steps=[
    ('prep', preprocessor),
    ('ml', model)
])

pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['age', 'annual_income',
                                                   'dependents', 'tenure',
                                                   'senior_citizen',
                                                   'monthlycharges',
                                                   'totalcharges',
                                                   'num_services',
                                                   'has_phone_service',
                                                   'has_internet_service',
                                                   'has_online_security',
                                                   'has_online_backup',
                                                   'ha...
                                                   'num_service_calls',
                                                   'late_payments',
                                                   'avg_monthly_gb',
                                                   'days_since_last_interaction',
                                                   'credit_score']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'marital_status', 'contract',
                                                   'payment_method',
                                                   'paperless_billing'])])),
                ('ml', LogisticRegression(max_iter=1000))])

In [65]:
# Predict churn labels for the held-out test set.
y_pred = pipe.predict(X_test)

In [66]:
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [67]:
# Import metrics to evaluate the model on data it did not train on.
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [68]:
# Accuracy is the fraction of all predictions that are correct.
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.899815


In [69]:
# Confusion matrix layout for binary classification:
# [[true negatives, false positives],
#  [false negatives, true positives]]
print(confusion_matrix(y_test, y_pred))

# Precision, recall and F1-score help show how well the model identifies churners.
print(classification_report(y_test, y_pred, zero_division=0))

[[179904     60]
 [ 19977     59]]
              precision    recall  f1-score   support

           0       0.90      1.00      0.95    179964
           1       0.50      0.00      0.01     20036

    accuracy                           0.90    200000
   macro avg       0.70      0.50      0.48    200000
weighted avg       0.86      0.90      0.85    200000



## 3. How I will interpret the results

- **Accuracy:** overall percentage of correct predictions.
- **Precision for churn (class 1):** of the customers predicted to churn, how many actually churned.
- **Recall for churn (class 1):** of the customers who actually churned, how many the model found.
- **F1-score:** balances precision and recall.
- **Confusion matrix:** shows the correct and incorrect predictions for each class.

If the dataset has many more non-churn customers than churn customers, accuracy alone can look good even when the model misses churners. I will therefore pay close attention to class 1 precision, recall and F1-score.

## 4. Next improvement

This is a first baseline model. After reviewing the results, I can compare it with a tree-based model and check whether class weighting improves detection of churn customers. I should compare models using the same train/test split and the same evaluation metrics.
